In [3]:
import ee
import geemap

# Authenticate — this opens a login flow the first time (one-time per session)
ee.Authenticate()

# Initialize the Earth Engine library with your project
ee.Initialize(project='white-script-288218')  # we'll get this ID in the next step

In [4]:
!pip install geemap -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 71.2 MB/s eta 0:00:00


In [5]:
import ee
import geemap

# Define AOI as a rectangle around the Raiwind Road corridor (southwest Lahore)
# Coordinates are [longitude, latitude] pairs — this box covers roughly
# Thokar Niaz Baig down to the Al-Kabir/Etihad Town cluster
aoi = ee.Geometry.Rectangle([74.1575, 31.34, 74.3075, 31.48])

# Quick visual check — display the AOI boundary on a map
Map = geemap.Map(center=[31.41, 74.25], zoom=12)
Map.addLayer(aoi, {'color': 'red'}, 'AOI Boundary')
Map

Map(center=[31.41, 74.25], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright'…

In [6]:
def get_sentinel2_composite(aoi, start_date, end_date, cloud_threshold=20):
    """
    Loads Sentinel-2 imagery for the given AOI and date range, filters by
    cloud cover, and creates a single cloud-reduced median composite image.
    """
    collection = (
        ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')  # Sentinel-2 Surface Reflectance data
        .filterBounds(aoi)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', cloud_threshold))
    )

    print(f"Number of images found ({start_date} to {end_date}):", collection.size().getInfo())

    # median() takes the middle value per pixel across all images —
    # this naturally removes clouds/outliers since clouds rarely appear
    # in the same exact pixel location across multiple different image dates
    composite = collection.median().clip(aoi)
    return composite


# 2019 composite — using a wider date window to ensure enough cloud-free images exist
image_2019 = get_sentinel2_composite(aoi, '2019-01-01', '2019-12-31')

# 2025 composite
image_2025 = get_sentinel2_composite(aoi, '2025-01-01', '2025-12-31')

Number of images found (2019-01-01 to 2019-12-31): 67
Number of images found (2025-01-01 to 2025-12-31): 88


In [7]:
# Visualization parameters for natural-color RGB display
rgb_vis = {
    'bands': ['B4', 'B3', 'B2'],  # Red, Green, Blue bands
    'min': 0,
    'max': 3000,
}

Map = geemap.Map(center=[31.41, 74.23], zoom=12)
Map.addLayer(image_2019, rgb_vis, 'RGB 2019')
Map.addLayer(image_2025, rgb_vis, 'RGB 2025')
Map.addLayer(aoi, {'color': 'red'}, 'AOI Boundary', opacity=0.3)
Map

Map(center=[31.41, 74.23], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright'…

In [8]:
def compute_ndvi(image):
    """
    NDVI = (NIR - Red) / (NIR + Red)
    Sentinel-2 bands: B8 = Near-Infrared (NIR), B4 = Red
    Healthy vegetation reflects NIR strongly and absorbs Red strongly,
    so high NDVI (close to +1) indicates vegetation/crops.
    """
    return image.normalizedDifference(['B8', 'B4']).rename('NDVI')

ndvi_2019 = compute_ndvi(image_2019)
ndvi_2025 = compute_ndvi(image_2025)

ndvi_vis = {'min': -0.2, 'max': 0.8, 'palette': ['red', 'yellow', 'green']}

Map2 = geemap.Map(center=[31.41, 74.23], zoom=12)
Map2.addLayer(ndvi_2019, ndvi_vis, 'NDVI 2019')
Map2.addLayer(ndvi_2025, ndvi_vis, 'NDVI 2025')
Map2

Map(center=[31.41, 74.23], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright'…

In [9]:
def compute_ndbi(image):
    """
    NDBI = (SWIR - NIR) / (SWIR + NIR)
    Sentinel-2 bands: B11 = Short-Wave Infrared (SWIR), B8 = NIR
    Built-up surfaces (concrete, roads, roofs) reflect SWIR more than NIR,
    so high NDBI indicates urban/built-up land.
    """
    return image.normalizedDifference(['B11', 'B8']).rename('NDBI')

ndbi_2019 = compute_ndbi(image_2019)
ndbi_2025 = compute_ndbi(image_2025)

ndbi_vis = {'min': -0.5, 'max': 0.3, 'palette': ['blue', 'white', 'red']}

Map3 = geemap.Map(center=[31.41, 74.23], zoom=12)
Map3.addLayer(ndbi_2019, ndbi_vis, 'NDBI 2019')
Map3.addLayer(ndbi_2025, ndbi_vis, 'NDBI 2025')
Map3

Map(center=[31.41, 74.23], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright'…

In [10]:
# Generate 2000 random points scattered across our AOI
sample_points = ee.FeatureCollection.randomPoints(region=aoi, points=2000, seed=42)

# Combine 2025 image with its NDVI and NDBI bands (only useful bands, no empty QA bands)
feature_bands = ['B2', 'B3', 'B4', 'B8', 'B11', 'B12', 'NDVI', 'NDBI']
combined_2025 = image_2025.select(['B2','B3','B4','B8','B11','B12']) \
    .addBands(ndvi_2025).addBands(ndbi_2025)

# Extract pixel values at each random point
sampled = combined_2025.sampleRegions(
    collection=sample_points,
    scale=10,
    geometries=True,
    tileScale=4
)

print("Total sampled points:", sampled.size().getInfo())

Total sampled points: 2000


In [11]:
# Only keep points where the signal is UNAMBIGUOUS — this is what makes
# threshold-based labeling trustworthy: we discard uncertain/borderline points
# rather than risk mislabeling them.

vegetation_points = sampled.filter(ee.Filter.And(
    ee.Filter.gt('NDVI', 0.5),    # strongly vegetated
    ee.Filter.lt('NDBI', -0.1)    # clearly not built-up
)).map(lambda f: f.set('class', 0))

builtup_points = sampled.filter(ee.Filter.And(
    ee.Filter.gt('NDBI', 0.1),    # strongly built-up
    ee.Filter.lt('NDVI', 0.2)     # clearly not vegetated
)).map(lambda f: f.set('class', 1))

other_points = sampled.filter(ee.Filter.And(
    ee.Filter.gt('NDVI', -0.1),
    ee.Filter.lt('NDVI', 0.2)
)).filter(ee.Filter.And(
    ee.Filter.gt('NDBI', -0.1),
    ee.Filter.lt('NDBI', 0.05)
)).map(lambda f: f.set('class', 2))

print("Vegetation samples:", vegetation_points.size().getInfo())
print("Built-up samples:", builtup_points.size().getInfo())
print("Other samples:", other_points.size().getInfo())

Vegetation samples: 168
Built-up samples: 264
Other samples: 241


In [12]:
# Take a balanced number from each class so no single class dominates training
training_points = (
    vegetation_points.limit(70)
    .merge(builtup_points.limit(70))
    .merge(other_points.limit(70))
)

print("Final training set size:", training_points.size().getInfo())

Final training set size: 210


In [13]:
# List of features (bands) the classifier will learn from
feature_bands = ['B2', 'B3', 'B4', 'B8', 'B11', 'NDVI', 'NDBI']

classifier = ee.Classifier.smileRandomForest(numberOfTrees=50).train(
    features=training_points,
    classProperty='class',
    inputProperties=feature_bands
)

print("Classifier trained successfully")

Classifier trained successfully


In [14]:
# Apply the trained classifier to full images (not just sample points)
classified_2019 = image_2019.addBands(compute_ndvi(image_2019)).addBands(compute_ndbi(image_2019)) \
    .select(feature_bands).classify(classifier)

classified_2025 = combined_2025.select(feature_bands).classify(classifier)

# Visualize the classified maps
class_vis = {'min': 0, 'max': 2, 'palette': ['green', 'red', 'yellow']}

Map4 = geemap.Map(center=[31.41, 74.23], zoom=13)
Map4.addLayer(classified_2019, class_vis, 'Classified 2019')
Map4.addLayer(classified_2025, class_vis, 'Classified 2025')
Map4

Map(center=[31.41, 74.23], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright'…

In [15]:
import geemap

Map4 = geemap.Map(center=[31.41, 74.23], zoom=13)
Map4.split_map(
    left_layer=geemap.ee_tile_layer(classified_2019, class_vis, '2019'),
    right_layer=geemap.ee_tile_layer(classified_2025, class_vis, '2025')
)
Map4

Map(center=[31.41, 74.23], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_o…

In [16]:
print("Vegetation:", vegetation_points.size().getInfo())
print("Built-up:", builtup_points.size().getInfo())
print("Other:", other_points.size().getInfo())
print("Total training:", training_points.size().getInfo())

Vegetation: 168
Built-up: 264
Other: 241
Total training: 210


In [17]:
# Combine both classified years into a single "change" image
# Each pixel gets a unique code: old_class * 10 + new_class
# e.g. 0 -> 1 (Vegetation to Built-up) becomes code 1
#      1 -> 1 (Built-up stayed Built-up) becomes code 11

change_map = classified_2019.multiply(10).add(classified_2025).rename('change')

# Visualize raw change codes (optional, just for sanity check)
change_vis = {'min': 0, 'max': 22, 'palette': ['white', 'orange', 'red', 'yellow', 'green', 'blue']}

In [18]:
# Class codes: 0 = Vegetation, 1 = Built-up, 2 = Other

veg_to_builtup = classified_2019.eq(0).And(classified_2025.eq(1))
veg_to_other = classified_2019.eq(0).And(classified_2025.eq(2))
builtup_to_veg = classified_2019.eq(1).And(classified_2025.eq(0))  # unlikely, but check
unchanged_veg = classified_2019.eq(0).And(classified_2025.eq(0))
unchanged_builtup = classified_2019.eq(1).And(classified_2025.eq(1))

Map6 = geemap.Map()
Map6.centerObject(aoi, zoom=13)
Map6.addLayer(veg_to_builtup.selfMask(), {'palette': ['red']}, 'Vegetation → Built-up')
Map6.addLayer(aoi, {'color': 'black'}, 'AOI boundary')
Map6

Map(center=[31.410004425814954, 74.23249999999794], controls=(WidgetControl(options=['position', 'transparent_…

In [19]:
# Pixel area in square meters (Sentinel-2 = 10m resolution, so each pixel = 100 sq m)
pixel_area = ee.Image.pixelArea()

def calculate_area(classified_image, class_value, region):
    mask = classified_image.eq(class_value)
    area_image = mask.multiply(pixel_area).rename('area')  # explicit band name
    area_stats = area_image.reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=region,
        scale=10,
        maxPixels=1e9
    )
    return area_stats.getInfo()

# Calculate for 2019
veg_area_2019 = calculate_area(classified_2019, 0, aoi)
builtup_area_2019 = calculate_area(classified_2019, 1, aoi)
other_area_2019 = calculate_area(classified_2019, 2, aoi)

# Calculate for 2025
veg_area_2025 = calculate_area(classified_2025, 0, aoi)
builtup_area_2025 = calculate_area(classified_2025, 1, aoi)
other_area_2025 = calculate_area(classified_2025, 2, aoi)

# Convert sq meters to hectares (1 hectare = 10,000 sq m)
def sqm_to_hectares(area_dict):
    return round(area_dict['area'] / 10000, 2)

print("=== 2019 ===")
print("Vegetation:", sqm_to_hectares(veg_area_2019), "ha")
print("Built-up:", sqm_to_hectares(builtup_area_2019), "ha")
print("Other:", sqm_to_hectares(other_area_2019), "ha")

print("\n=== 2025 ===")
print("Vegetation:", sqm_to_hectares(veg_area_2025), "ha")
print("Built-up:", sqm_to_hectares(builtup_area_2025), "ha")
print("Other:", sqm_to_hectares(other_area_2025), "ha")

print("\n=== Net Change (2019 → 2025) ===")
print("Vegetation change:", round(sqm_to_hectares(veg_area_2025) - sqm_to_hectares(veg_area_2019), 2), "ha")
print("Built-up change:", round(sqm_to_hectares(builtup_area_2025) - sqm_to_hectares(builtup_area_2019), 2), "ha")
print("Other change:", round(sqm_to_hectares(other_area_2025) - sqm_to_hectares(other_area_2019), 2), "ha")

=== 2019 ===
Vegetation: 5621.49 ha
Built-up: 3670.45 ha
Other: 12849.87 ha

=== 2025 ===
Vegetation: 5918.99 ha
Built-up: 4950.95 ha
Other: 11271.87 ha

=== Net Change (2019 → 2025) ===
Vegetation change: 297.5 ha
Built-up change: 1280.5 ha
Other change: -1578.0 ha


In [20]:
veg_to_builtup_area = calculate_area(veg_to_builtup, 1, aoi)
print("Vegetation → Built-up (direct conversion):", sqm_to_hectares(veg_to_builtup_area), "ha")

Vegetation → Built-up (direct conversion): 245.67 ha


In [21]:
# Generate 50 validation points, separate seed from training (so they don't overlap)
validation_points = ee.FeatureCollection.randomPoints(region=aoi, points=50, seed=99)

# Get the model's predicted class at each point (using 2025 classification)
validation_sampled = classified_2025.sampleRegions(
    collection=validation_points,
    scale=10,
    geometries=True
)

# Convert to a list we can inspect one-by-one
validation_list = validation_sampled.getInfo()['features']

print(f"Total validation points: {len(validation_list)}")
for i, point in enumerate(validation_list[:5]):
    coords = point['geometry']['coordinates']
    predicted_class = point['properties']['classification']
    print(f"Point {i+1}: Lon={coords[0]:.4f}, Lat={coords[1]:.4f}, Predicted class={predicted_class}")

Total validation points: 50
Point 1: Lon=74.1731, Lat=31.4210, Predicted class=2
Point 2: Lon=74.3010, Lat=31.4268, Predicted class=1
Point 3: Lon=74.2771, Lat=31.4171, Predicted class=2
Point 4: Lon=74.2891, Lat=31.3921, Predicted class=2
Point 5: Lon=74.2937, Lat=31.3762, Predicted class=1


In [22]:
class_names = {0: "Vegetation", 1: "Built-up", 2: "Other"}

print(f"Total validation points: {len(validation_list)}\n")
for i, point in enumerate(validation_list):
    lon, lat = point['geometry']['coordinates']
    predicted = class_names[point['properties']['classification']]
    maps_link = f"https://www.google.com/maps/place/{lat},{lon}/@{lat},{lon},18z"
    print(f"Point {i+1}: Predicted={predicted} | {maps_link}")

Total validation points: 50

Point 1: Predicted=Other | https://www.google.com/maps/place/31.42095759758318,74.17312944175738/@31.42095759758318,74.17312944175738,18z
Point 2: Predicted=Built-up | https://www.google.com/maps/place/31.426796646929958,74.301049538216/@31.426796646929958,74.301049538216,18z
Point 3: Predicted=Other | https://www.google.com/maps/place/31.417094841861466,74.27706452013001/@31.417094841861466,74.27706452013001,18z
Point 4: Predicted=Other | https://www.google.com/maps/place/31.392121676962944,74.28910194493722/@31.392121676962944,74.28910194493722,18z
Point 5: Predicted=Built-up | https://www.google.com/maps/place/31.376221496434027,74.29368335288623/@31.376221496434027,74.29368335288623,18z
Point 6: Predicted=Vegetation | https://www.google.com/maps/place/31.416555852690994,74.30060038057395/@31.416555852690994,74.30060038057395,18z
Point 7: Predicted=Vegetation | https://www.google.com/maps/place/31.417723662560352,74.17887865957576/@31.417723662560352,74.

In [23]:
validation_results = [
    {"point_id": 1, "correct": False, "note": "Predicted Other, actual Built-up (residential complex)"},
    {"point_id": 2, "correct": True, "note": ""},
    {"point_id": 3, "correct": False, "note": "Predicted Other, actual Vegetation (grass edge)"},
    {"point_id": 4, "correct": True, "note": ""},
    {"point_id": 5, "correct": True, "note": ""},
    {"point_id": 6, "correct": True, "note": ""},
    {"point_id": 7, "correct": True, "note": ""},
    {"point_id": 8, "correct": True, "note": ""},
    {"point_id": 9, "correct": True, "note": ""},
    {"point_id": 10, "correct": True, "note": ""},
    {"point_id": 11, "correct": False, "note": "Predicted Built-up, actual empty/bare plot inside a housing society"},
    {"point_id": 12, "correct": True, "note": ""},
    {"point_id": 13, "correct": True, "note": ""},
    {"point_id": 14, "correct": True, "note": ""},
    {"point_id": 15, "correct": True, "note": ""},
    {"point_id": 16, "correct": True, "note": ""},
    {"point_id": 17, "correct": True, "note": ""},
    {"point_id": 18, "correct": True, "note": ""},
    {"point_id": 19, "correct": True, "note": ""},
    {"point_id": 20, "correct": False, "note": "Predicted Other, actual tree cover inside a residential house (mixed pixel)"},
    {"point_id": 21, "correct": True, "note": ""},
    {"point_id": 22, "correct": False, "note": "Predicted Other, actual Vegetation"},
    {"point_id": 23, "correct": True, "note": ""},
    {"point_id": 24, "correct": True, "note": ""},
    {"point_id": 25, "correct": True, "note": ""},
    {"point_id": 26, "correct": True, "note": ""},
    {"point_id": 27, "correct": True, "note": ""},
    {"point_id": 28, "correct": False, "note": "Mixed pixel near built-up boundary, point slightly off from adjacent structure"},
    {"point_id": 29, "correct": True, "note": ""},
    {"point_id": 30, "correct": True, "note": ""},
    {"point_id": 31, "correct": False, "note": "Point landed on road within built-up area, predicted Other instead of Built-up"},
    {"point_id": 32, "correct": True, "note": ""},
    {"point_id": 33, "correct": True, "note": ""},
    {"point_id": 34, "correct": True, "note": ""},
    {"point_id": 35, "correct": True, "note": ""},
    {"point_id": 36, "correct": True, "note": ""},
    {"point_id": 37, "correct": True, "note": ""},
    {"point_id": 38, "correct": False, "note": "Predicted Vegetation, actual bare/empty plot inside a housing society"},
    {"point_id": 39, "correct": False, "note": "Predicted Other, actual Built-up"},
    {"point_id": 40, "correct": True, "note": ""},
    {"point_id": 41, "correct": True, "note": ""},
    {"point_id": 42, "correct": True, "note": ""},
    {"point_id": 43, "correct": True, "note": ""},
    {"point_id": 44, "correct": True, "note": ""},
    {"point_id": 45, "correct": False, "note": "Predicted Other, actual Vegetation"},
    {"point_id": 46, "correct": True, "note": "New society, empty/bare plots — correctly matched Other"},
    {"point_id": 47, "correct": True, "note": ""},
    {"point_id": 48, "correct": True, "note": ""},
    {"point_id": 49, "correct": True, "note": ""},
    {"point_id": 50, "correct": True, "note": ""},
]

import pandas as pd
df = pd.DataFrame(validation_results)
accuracy = df['correct'].mean() * 100
print(f"Validation accuracy: {accuracy:.1f}%")
df.to_csv('validation_table.csv', index=False)
print("Saved validation_table.csv")

Validation accuracy: 80.0%
Saved validation_table.csv


In [24]:
# Generate just 10 extra points (quick top-up, not a full new batch)
extra_points = ee.FeatureCollection.randomPoints(region=aoi, points=10, seed=7)

extra_sampled = classified_2025.sampleRegions(
    collection=extra_points,
    scale=10,
    geometries=True
)

extra_list = extra_sampled.getInfo()['features']
class_names = {0: "Vegetation", 1: "Built-up", 2: "Other"}

for i, point in enumerate(extra_list):
    lon, lat = point['geometry']['coordinates']
    predicted = class_names[point['properties']['classification']]
    link = f"https://www.google.com/maps/place/{lat},{lon}/@{lat},{lon},18z"
    print(f"Extra Point {i+1}: Predicted={predicted} | {link}")

Extra Point 1: Predicted=Built-up | https://www.google.com/maps/place/31.468119149999456,74.2739204166356/@31.468119149999456,74.2739204166356,18z
Extra Point 2: Predicted=Built-up | https://www.google.com/maps/place/31.40667438456568,74.19019743215566/@31.40667438456568,74.19019743215566,18z
Extra Point 3: Predicted=Vegetation | https://www.google.com/maps/place/31.476114156028117,74.2642186115671/@31.476114156028117,74.2642186115671,18z
Extra Point 4: Predicted=Built-up | https://www.google.com/maps/place/31.431378054878966,74.30509195699454/@31.431378054878966,74.30509195699454,18z
Extra Point 5: Predicted=Other | https://www.google.com/maps/place/31.40200314508826,74.2198418365316/@31.40200314508826,74.2198418365316,18z
Extra Point 6: Predicted=Vegetation | https://www.google.com/maps/place/31.376041833377204,74.19397035634896/@31.376041833377204,74.19397035634896,18z
Extra Point 7: Predicted=Other | https://www.google.com/maps/place/31.362836598700646,74.18184310001335/@31.3628365

In [25]:
# Generate 20 more points, different seed so they're new
extra_points_2 = ee.FeatureCollection.randomPoints(region=aoi, points=20, seed=15)

extra_sampled_2 = classified_2025.sampleRegions(
    collection=extra_points_2,
    scale=10,
    geometries=True
)

extra_list_2 = extra_sampled_2.getInfo()['features']
class_names = {0: "Vegetation", 1: "Built-up", 2: "Other"}

for i, point in enumerate(extra_list_2):
    lon, lat = point['geometry']['coordinates']
    predicted = class_names[point['properties']['classification']]
    link = f"https://www.google.com/maps/place/{lat},{lon}/@{lat},{lon},18z"
    print(f"Batch2 Point {i+1}: Predicted={predicted} | {link}")

Batch2 Point 1: Predicted=Vegetation | https://www.google.com/maps/place/31.45132065418642,74.16001403860925/@31.45132065418642,74.16001403860925,18z
Batch2 Point 2: Predicted=Other | https://www.google.com/maps/place/31.38367751329222,74.16522426725713/@31.38367751329222,74.16522426725713,18z
Batch2 Point 3: Predicted=Built-up | https://www.google.com/maps/place/31.450602001959126,74.30383431559677/@31.450602001959126,74.30383431559677,18z
Batch2 Point 4: Predicted=Built-up | https://www.google.com/maps/place/31.475126009215586,74.30033088598871/@31.475126009215586,74.30033088598871,18z
Batch2 Point 5: Predicted=Other | https://www.google.com/maps/place/31.42841361444137,74.24508449601537/@31.42841361444137,74.24508449601537,18z
Batch2 Point 6: Predicted=Built-up | https://www.google.com/maps/place/31.464346225806153,74.23313690273658/@31.464346225806153,74.23313690273658,18z
Batch2 Point 7: Predicted=Vegetation | https://www.google.com/maps/place/31.341905852580663,74.293413858301/@3